#### By Sugandha Sharma, April 2024. 
#### Contact sugandhas@nvidia.com for questions.

# Data Curation for DAPT (Domain Adaptive Pre-Training)

This notebook walks through the processing pipeline for data curation required for DAPT. Given a directory containing a set of folders and files, this pipeline performs data collection and cleaning. In this notebook, we will use the datasets in the `dapt-data-curation/datasets` folder to illustrate data curation through this pipeline.

The notebook follows the steps below:<br>
- Step 1: Install requirements<br>
- Step 2: Find all text files in the given directory tree<br>
- Step 3: Examine the identified files<br>
- Step 4: Filter files by file type<br>
- Step 5: Copy all files we want to include and compress them<br>
- Step 6: Check for duplicate files<br>
- Step 7: Remove duplicate files<br>

#### The complete pipeline is illustrated in the schematic shown below:

![pipeline](imgs/pipeline.png)


## Step 1: Install requirements

In [12]:
# install all required packages for data curation
! pip3 install -r ../requirements.txt

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 653.6/653.6 kB 64.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 145.7/145.7 kB 237.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 113.4 MB/s eta 0:00:00a 0:00:01
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.9/58.9 kB 301.9 MB/s eta 0:00:00


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.0/57.0 kB 294.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.1/48.1 kB 282.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.4/41.4 kB 281.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.2/61.2 kB 302.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 332.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.3/155.3 kB 343.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 441.8/441.8 kB 173.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.3/199.3 kB 352.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.3/300.3 kB 122.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 122.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 118.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.6/74.6 kB 312.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 117.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 341.8/341.8 kB 217.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.9/140.9 kB 213.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 152.9/152.9 kB 198.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.0/247.0 kB 214.5 MB/s eta 0:00:00
  Created wheel for cchardet: filename=cchardet-2.1.7-cp310-cp310-linux_x86_64.whl size=289380 sha256=cea25ee50096937c2bbf2aad96ecec8ab9f54652e5824c4fece1c24ac42007fc
  Stored in directory: /tmp/pip-ephem-wheel-cache-0srrsbif/wheels/ee/e0/ab/e01326f15c59438d080b1496dbab8091e952ec72f35e3c437e
  Created wheel for docx2txt: filename=docx2txt-0.8-py3-none-any.whl size=3960 sha256=4bd0b34e43d90c0d93414dd312cd92e2f7555ed96d933cfd94fafab0ef317cae
  Stored in directory: /tmp/pip-ephem-wheel-cache-0srrsbif/wheels/22/58/cf/093d0a6c3ecfdfc5f6ddd5524043b88e59a9a199cb02352966
  Created wheel for python-p

## Step 2: Find all text files in the given directory tree

Now we will run the `create_manifest.py` script that takes in a directory as input, finds all the text files in the directory tree, and produces a manifest csv file containing a list of file names (file paths), file sizes, and number of lines in each of the text files found in the directory tree. 
- Input1: `directory`: the path to the input collection directory that has the data we want to search through
- Input2: `manifest_file`: the path to the output csv file
- output: `manifest.csv`: a csv file with three columns: Path (listing file names), Size (listing file sizes) and Lines (listing number of lines in each file) is saved to the  `dapt-data-curation/output` folder

#### An exmaple of a sample output showing the manifest csv file is shown below. 

![manifest](imgs/manifest_csv.png)

In [5]:
# specify path to the input directory that has the data that we want to search through
directory = "../../../datasets/github_repos/"

In [6]:
# create a folder for storing output files under the dapt-data-curation directory
! mkdir ../../../output

mkdir: cannot create directory ‘../../../output’: File exists


In [7]:
# specify the path to (and the name of) the output csv file
manifest_file  = "../../../output/manifest.csv"

In [9]:
# create the manifest csv file with a list of all text files found. This csv will have three columns listing file names, their sizes and number of lines in each of them
! bash ../create_manifest.sh $directory $manifest_file

Done! Results saved to ../../../output/manifest.csv


#### This is what the `dapt-data-curation/output` folder should look like at the end of Step 2:

![pipeline](imgs/output_step2.png)

## Step3 (optional): Examine the identified files

Here, we will run `report_manifest.py` that enables us to examine the largest files from the indetified files listed in the manifest csv. It takes the manifest csv as input and reports the top 10 biggest files along with their corresponding percentile sizes. It allows visualization through text based histograms for various file types. 

- Input: `manifest.csv`: a csv file with three columns: Path (listing file names), Size (listing file sizes) and Lines (listing number of lines in each file)
- Output:<br> 
       - Top 10 biggest files along with their corresponding percentile sizes <br>
       - List of the top 10 biggest files with their paths, number of lines and file size<br>
       - List of files types greater than 5MB in size and number of such files for each file type <br>
       - Histograms for various file types showing the distribution of file sizes
- Output: `file_size_histogram.png`: saves the file size histogram in the `dapt-data-curation/output` folder

#### An example of a file size histogram output is shown below:

![histogram](imgs/file_size_histogram.png)



In [13]:
# run the report_manifest.py script, with the manifest csv file as an argument passed to the script
! python3 ../report_manifest.py $manifest_file

9847 Files
Top 10 percentiles (Size_MB):
0.90    0.008255
0.91    0.009185
0.92    0.010418
0.93    0.011559
0.94    0.012876
0.95    0.015491
0.96    0.021337
0.97    0.033552
0.98    0.061710
0.99    1.091250
Name: Size_MB, dtype: float64
Top 10 files (size):
/workspace/datasets/github_repos/spatial-lang/spatial/core/resources/chiselgen/template-level/fringeDE1SoC/Computer_System.sopcinfo : 127518 Lines 3.726355MB
/workspace/datasets/github_repos/spatial-lang/spatial/core/resources/chiselgen/template-level/fringeDE1SoC/Computer_System/synthesis/Computer_System.debuginfo : 100996 Lines 3.765121MB
/workspace/datasets/github_repos/spatial-lang/spatial/core/resources/chiselgen/template-level/fringeDE1SoC/CS_bak/Computer_System.xml : 22294 Lines 5.425511MB
/workspace/datasets/github_repos/spatial-lang/bin/arria10_debuggers/testbenches/arria10Test : 103268 Lines 5.681981MB
/workspace/datasets/github_repos/spatial-lang/spatial/core/resources/chiselgen/template-level/fringeDE1SoC/Computer_Sy


Common 4-grams in paths (occurring at least 100 times):
/workspace/datasets/github_repos: 9847 occurrences
workspace/datasets/github_repos/skywater-pdk-libs-sky130_fd_sc_lp: 6701 occurrences
datasets/github_repos/skywater-pdk-libs-sky130_fd_sc_lp/cells: 6553 occurrences
workspace/datasets/github_repos/spatial-lang: 2192 occurrences
datasets/github_repos/spatial-lang/spatial: 2122 occurrences
github_repos/spatial-lang/spatial/core: 2120 occurrences
spatial-lang/spatial/core/resources: 1777 occurrences
spatial/core/resources/chiselgen: 1505 occurrences
core/resources/chiselgen/template-level: 1488 occurrences
resources/chiselgen/template-level/fringeArria10: 702 occurrences
chiselgen/template-level/fringeArria10/build: 696 occurrences
template-level/fringeArria10/build/ip: 646 occurrences
workspace/datasets/github_repos/oh-lib: 635 occurrences
resources/chiselgen/template-level/fringeDE1SoC: 494 occurrences
fringeArria10/build/ip/ghrd_10as066n2: 463 occurrences
spatial-lang/spatial/core

### This is what the `dapt-data-curation/output` folder should look like at the end of Step 3:

![output_3](imgs/output_step3.png)

## Step4: Filter files by file type

Here we will run `process_manifest.py` to filter the files listed in the manifest csv. This will categorize the files based on their file types and generate separate csv files for each of the categories.

- Input: `manifest.csv`: a csv file with three columns: Path (listing file names), Size (listing file sizes) and Lines (listing number of lines in each file)
- Output: multiple filtered manifest csv files with one file for each of the following categories e.g., `manifest_filtered_cpp.csv`. The filtered csv files are saved in the `dapt-data-curation/output` folder

Following the are categories that the files are split into based on the file extensions: <br>

| **Category** | **File Extensions** |
|-------------------|---------------------|
| Viva              | ['.VX', '.VXH']     |
| VerilogVHDL       | ['.V', '.VH', '.VHDL'] |
| CPP               | ['.C', '.CPP', '.H', '.HPP'] |
| Python            | ['.PY']             |
| SV                | ['.SV']             |
| GV                | ['.GV']             |
| Config            | ['.CONFIG']         |
| Makefile          | ['Makefile', 'Makeppfile', '.mk'] |
| Perl              | ['.PM', '.PL']      |
| Tcl               | ['.TCL']            |
| Spec              | ['.spec']           |
| Yaml              | ['.yaml', '.yml']   |
| Spice             | ['.sp', '.cir', '.cmd', '.spf'] |
| VerilogAnalog     | ['.va']             |
| Skill             | ['.il']             |
| PT                | [".ptsh", ".proc"]  |
| EP3               | [".ep3", ".ep3.pp"] |
| Script            | ['NO_EXTENSION']    |



In [15]:
# run the process_manifest.py script, with the manifest csv file as an argument passed to the script
! python3 ../process_manifest.py $manifest_file


# VerilogVHDL Summary (['.V', '.VH', '.VHDL'])
	Total number of files: 3739
	Total Size: 10 MB (uncompressed)

# VerilogVHDLGeneratedOther Summary (['.V', '.VH', '.VHDL'])
	Total number of files: 60
	Total Size: 3 MB (uncompressed)

# CPP Summary (['.C', '.CPP', '.H', '.HPP'])
	Total number of files: 449
	Total Size: 4 MB (uncompressed)

# CPPGeneratedOther Summary (['.C', '.CPP', '.H', '.HPP'])
	Total number of files: 1
	Total Size: 0 MB (uncompressed)

# Python Summary (['.PY'])
	Total number of files: 18
	Total Size: 0 MB (uncompressed)

# SV Summary (['.SV'])
	Total number of files: 234
	Total Size: 3 MB (uncompressed)

# Config Summary (['.CONFIG'])
	Total number of files: 0
	Total Size: 0 MB (uncompressed)

# Makefile Summary (['Makefile', 'Makeppfile', '.mk'])
	Total number of files: 42
	Total Size: 0 MB (uncompressed)

# Perl Summary (['.PM', '.PL'])
	Total number of files: 1
	Total Size: 0 MB (uncompressed)

# Tcl Summary (['.TCL'])
	Total number of files: 47
	Total Size: 0 M

#### This is what the `dapt-data-curation/output` folder should look like at the end of Step 4:

![output_4](imgs/output_step4.png)

## Step 5: Copy all files we want to include and compress them

Here we will run `execute_manifest.py` which takes a manifest csv as input and copies over all the files listed in the csv, gzips them and calculates the md5sum. We can either use the original manifest csv if we want to include all the text files we found in the directory tree for DAPT training. 

Alternatively, we can also use any of the filtered manifest csv files generated in the previous step in we want to selectivley use a particular category of files for DAPT training. 

In this example, we just use the original manifest csv to illustrate processing of a larger dataset. 

- Input: `manifest.csv`: a csv file with three columns: Path (listing file names), Size (listing file sizes) and Lines (listing number of lines in each file)
- Output: manifest folder containing each file listed in the input csv gzipped (.gz). In addition this folder contains a .txt file listed the origianl paths from where each of these files was copied. The manifest folder is saved in the `dapt-data-curation/output` folder.

In [16]:
# run the process_manifest.py script, with the manifest csv file as an argument passed to the script
! python3 ../execute_manifest.py $manifest_file

Done.


#### This is what the `dapt-data-curation/output` folder should look like at the end of Step 4:

![output_5](imgs/output_step5.png)

#### This is what the `dapt-data-curation/output/manifest` folder should look like at the end of Step 4:

![output_5_manifest](imgs/manifest_folder.png)

#### This is what the `dapt-data-curation/output/manifest/info.txt` file should look like at the end of Step 4:

![output_5_manifest_info](imgs/manifest_folder_txt.png)

## Step 6: Check for duplicate files 

Here we run `report_exact_dedup.py` in order to check for duplicate files within a specific category of files. 

- For illustration, we will first run `execute_manifest.py` on a particular category of files by using the corresponding filtered manifest file generated in Step 4. This will create a `manifest_filtered_<category_name>` folder in `dapt-data-curation/output` folder whoose contents should be similar to the output of Step 5 shown above. 


- We will then take the `info.txt` file from `manifest_filtered_<category_name>` folder and provide it as input to `report_exact_dedup.py` to check whether there are any duplicte files within this category.

- Any duplicate files found will be printed out with a list of all duplicate file paths. If the script executes silently, this implies that there are no duplicate files present.

### Example 1

In [17]:
# Run execute_manifest.py on a manifest_filtered_cpp.csv generated in Step 4 to generate the filtered_manifest folder with gzipped files of type ['.C', '.CPP', '.H', '.HPP']
# This will output a manifest_filtered_cpp folder in dapt-data-curation/output directory
filtered_manifest = "../../../output/manifest_filtered_cpp.csv"
! python3 ../execute_manifest.py $filtered_manifest

Done.


In [19]:
# Now we will run report_exact_dedup.py to check for duplicates of files of type ['.C', '.CPP', '.H', '.HPP']
# All duplicates will be printed. Empty output implies there are no duplicate files.
info_file = "../../../output/manifest_filtered_cpp/info.txt"
! python3 ../report_exact_dedup.py $info_file

### Example 2

In [20]:
# Run execute_manifest.py on a manifest_filtered_verilogvhdl.csv generated in Step 4 to generate the filtered_manifest folder with gzipped files of type ['.V', '.VH', '.VHDL']
# This will output a manifest_filtered_verilogvhdl folder in dapt-data-curation/output directory
filtered_manifest = "../../../output/manifest_filtered_verilogvhdl.csv"
! python3 ../execute_manifest.py $filtered_manifest

Done.


In [21]:
# Now we will run report_exact_dedup.py to check for duplicates of files of type ['.V', '.VH', '.VHDL']
# All duplicates will be printed. Empty output implies there are no duplicate files.
info_file = "../../../output/manifest_filtered_verilogvhdl/info.txt"
! python3 ../report_exact_dedup.py $info_file